# Search Capabilities — No Server Required

Spicy Regs publishes all data as public files on Cloudflare R2. That means you can do mirrulations-search-style queries (and more) from any Python shell without running a backend.

This notebook walks through the three layers of search available today:

| Layer | Use when | Scope | Latency |
|---|---|---|---|
| 1. Prebuilt docket index (`docket_search.json.gz`) | You want the same fast metadata search the site uses | Docket title + abstract (276K) | One ~10 MB download, then in-memory |
| 2. DuckDB over `dockets.parquet` | You need SQL flexibility on docket metadata | Docket title + abstract + fields | <1s per query |
| 3. DuckDB over the per-agency `comments/` tree | Full-text over comment bodies (the mirrulations-search equivalent) | 25M+ comments | A few seconds, faster if you narrow by docket |

Everything below hits public URLs. No auth, no local downloads of multi-GB files.

## Setup

In [1]:
import duckdb

# Custom R2 domain — avoids the rate limits on the pub-*.r2.dev dev endpoint.
import os
R2_BASE_URL = (
    os.environ.get("SPICY_REGS_R2_URL")
    or os.environ.get("R2_PUBLIC_URL")
    or f"https://data.{os.environ.get('SPICYREGS_DOMAIN') or 'spicy-regs.dev'}"
).rstrip("/")

conn = duckdb.connect()
conn.execute("INSTALL httpfs; LOAD httpfs;")
print("ready — all queries will stream directly from R2")

ready — all queries will stream directly from R2


## Layer 1 — Prebuilt docket search index

This is the exact blob the `/search` page on the live site loads into MiniSearch. It's a gzipped JSON with every docket's `id`, `agency_code`, `title`, `docket_type`, `modify_date`, and `abstract` (fields abbreviated as `id/a/t/x/d/s` for payload size — see `src/spicy_regs/transforms/build_search_index.py`).

**When to reach for this:** you want docket-level fuzzy/prefix search with zero warm-up cost per query. Load once, then filter in pure Python.

The site uses [`minisearch`](https://github.com/lucaong/minisearch) for ranked fuzzy/prefix matching — the cell below keeps it simple with plain substring filtering so there's nothing to install. For production-quality ranking, feed these docs into your text-search lib of choice (Whoosh, Tantivy, even BM25 via `rank_bm25`).

In [2]:
import gzip
import json
import time
import urllib.error
import urllib.request
from pathlib import Path

INDEX_URL = f"{R2_BASE_URL}/docket_search.json.gz"
CACHE_PATH = Path(".cache/docket_search.json.gz")

# r2.dev public buckets rate-limit dev traffic (429). Cache locally so
# re-running this notebook doesn't re-hit R2, and retry with backoff on
# the first fetch. Delete .cache/docket_search.json.gz to force a refresh.
def _fetch_with_retry(url: str, attempts: int = 4) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": "spicy-regs-notebook"})
    for i in range(attempts):
        try:
            with urllib.request.urlopen(req) as resp:
                return resp.read()
        except urllib.error.HTTPError as e:
            if e.code != 429 or i == attempts - 1:
                raise
            wait = 2 ** i
            print(f"  429 rate-limited, retrying in {wait}s…")
            time.sleep(wait)
    raise RuntimeError("unreachable")

if CACHE_PATH.exists():
    raw = CACHE_PATH.read_bytes()
    print(f"using cached index at {CACHE_PATH}")
else:
    print(f"fetching {INDEX_URL}…")
    raw = _fetch_with_retry(INDEX_URL)
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CACHE_PATH.write_bytes(raw)

# R2 may or may not auto-decompress. Detect the gzip magic bytes (1f 8b).
if raw[:2] == b"\x1f\x8b":
    raw = gzip.decompress(raw)
payload = json.loads(raw)

print(f"index version: {payload['version']}")
print(f"dockets indexed: {payload['count']:,}")
payload['docs'][0]

using cached index at .cache/docket_search.json.gz


index version: 2026-08-12T23:51:06Z


dockets indexed: 276,308


{'id': 'FWS-R4-ES-2026-1750',
 'a': 'FWS',
 't': 'Receipt of Incidental Take Permit Application and Proposed Habitat Conservation Plan for Sand Skink; Lake County, FL; Categorical Exclusion (Casa del Rosa)',
 'x': 'Nonrulemaking',
 'd': '2026-07-02T20:46:17Z'}

In [3]:
def search_docket_index(query: str, agency: str | None = None, limit: int = 10):
    """Plain substring search over the prebuilt index.

    Fields: id, a=agency, t=title, x=docket_type, d=modify_date, s=abstract.
    Swap this for minisearch/whoosh/BM25 if you need ranking.
    """
    q = query.lower()
    hits = []
    for doc in payload['docs']:
        if agency and doc.get('a') != agency:
            continue
        if q in doc.get('t', '').lower() or q in doc.get('s', '').lower():
            hits.append(doc)
            if len(hits) >= limit:
                break
    return hits

for hit in search_docket_index("biotech", limit=5):
    print(f"[{hit['a']}] {hit['id']} — {hit['t']}")

[EPA] EPA-HQ-OPP-2016-0260 — Notice of Receipt - New Active Ingredient 1-Triacontanol for Registration as EPA Reg. No. 90866-EN
[FDA] FDA-2022-D-2512 — Q5A(R2) Viral Safety Evaluation of Biotechnology Products Derived from Cell Lines of Human or Animal Origin; International Council for Harmonisation; Draft Guidance for Industry; Availability
[BIS] BIS-2024-0050 — Controls on Certain Laboratory Equipment and Related Technology to Address Dual Use Concerns about Biotechnology
[EPA] EPA-HQ-OPP-2024-0518 — Mesotrione- Pesticide Petition (2F9014)- Establish Tolerances for Mesotrione-resistant Soybean
[BIS] BIS-2024-0054 — Impact of the Implementation of the Chemical Weapons Convention (CWC) on Legitimate Commercial Chemical, Biotechnology, and Pharmaceutical Activities Involving “Schedule 1” Chemicals (including “Schedule 1” Chemicals produced as Intermediates) During Calendar Year 2024


## Layer 2 — DuckDB over `dockets.parquet`

When you need SQL. 276K rows, so `ILIKE` across title + abstract is plenty fast — DuckDB streams the Parquet file from R2 and pushes down the predicate.

**When to reach for this:** you want to combine keyword matching with richer filters (agency, date windows, docket type) or join results into a pandas DataFrame.

In [4]:
conn.execute(f"""
    SELECT docket_id, agency_code, docket_type, modify_date, title
    FROM read_parquet('{R2_BASE_URL}/dockets.parquet')
    WHERE (title ILIKE '%PFAS%' OR abstract ILIKE '%PFAS%')
      AND agency_code = 'EPA'
      AND modify_date >= '2024-01-01'
    ORDER BY modify_date DESC
    LIMIT 10
""").fetchdf()

,docket_id,agency_code,docket_type,modify_date,title
0,EPA-HQ-OW-2025-0654,EPA,Rulemaking,2026-07-02T20:16:20Z,Rescission of Regulatory Determinations and Re...
1,EPA-HQ-OLEM-2020-0527,EPA,Nonrulemaking,2026-07-02T20:16:18Z,Interim Guidance on the Destruction and Dispos...
2,EPA-HQ-OW-2025-1742,EPA,Rulemaking,2026-06-30T22:56:40Z,Extending the Compliance Deadline for the PFOA...
3,EPA-HQ-OW-2020-0582,EPA,Rulemaking,2026-06-29T13:52:27Z,Effluent Limitations Guidelines and Standards ...
4,EPA-HQ-OPPT-2020-0549,EPA,Rulemaking,2026-01-14T18:23:07Z,Reporting and Recordkeeping for Perfluoroalkyl...
5,EPA-HQ-TRI-2021-0049,EPA,Rulemaking,2025-12-29T20:27:42Z,NDAA Mandated Addition of Certain Per- and Pol...
6,EPA-HQ-TRI-2022-0270,EPA,Rulemaking,2025-12-29T20:27:17Z,Changes to Reporting Requirements for Per- and...
7,EPA-HQ-TRI-2022-0453,EPA,Rulemaking,2025-12-29T20:27:04Z,Implementing Statutory Addition of Certain Per...
8,EPA-HQ-OPPT-2023-0538,EPA,Rulemaking,2025-10-08T19:07:25Z,Addition of Certain Per- and Polyfluoroalkyl S...
9,EPA-HQ-OPPT-2023-0499,EPA,Nonrulemaking,2025-06-17T15:23:46Z,TSCA Section 4 Test Orders for the PFAS Nation...


## Layer 3 — Full-text over comments (partition-aware)

This is the mirrulations-search equivalent: search inside the 25M+ comment bodies.

### Anti-pattern: do not scan the flat file

There is a monolithic `comments.parquet` (~2.6 GB) meant for full-dataset analytics. **Don't use it for keyword search** — you'll burn bandwidth and minutes pulling data you throw away.

### Anti-pattern #2: the old fine-grained partition tree

Comments used to be Hive-partitioned all the way down to `comments/agency_code={A}/docket_id={D}/year={Y}/month={M}/part-0.parquet`, with a tiny `comments_index.parquet` rollup listing which partitions existed. **That tree stopped being written when the ETL moved comments onto the Iceberg catalog** (mid-April 2026) — `comments_index.parquet` is still refreshed from the catalog (so it happily lists row counts for docket/month combinations that are months old), but the physical partition files behind those newer entries were never created. Building partition URLs from `comments_index.parquet` today will 404 on any docket or month created after the cutover — silently missing recent comments rather than erroring loudly on all of them.

### Correct pattern: the canonical per-agency tree

The **live, rebuilt-daily** mirror is one Parquet file per agency: `comments/agency/agency_code={X}/part-0.parquet`. It's sorted by `(docket_id, posted_date)`, so DuckDB's Parquet reader prunes row groups on a `docket_id` filter and a docket-scoped query only pulls the bytes it needs — no partition discovery step required, just point at the one file for the agency you care about.

This mirrors the production pattern in `spicy-regs-ui/src/lib/duckdb/useDuckDBService.ts` (`buildCommentsSource`).

In [5]:
# Peek the canonical tree — one file per agency, so "does it exist" is just
# a HEAD-shaped query. Cheap because only the (small) docket_id column is read.
conn.execute(f"""
    SELECT docket_id, COUNT(*) AS comment_count
    FROM read_parquet('{R2_BASE_URL}/comments/agency/agency_code=EPA/part-0.parquet')
    GROUP BY docket_id
    ORDER BY comment_count DESC
    LIMIT 5
""").fetchdf()

,docket_id,comment_count
0,EPA-HQ-OA-2017-0190,63411
1,EPA-HQ-OAR-2025-0194,30933
2,EPA-HQ-OAR-2017-0355,26206
3,EPA-HQ-OA-2018-0259,22390
4,EPA-HQ-OAR-2006-0173,19452


In [6]:
def search_comments(
    phrase: str,
    agency: str,
    docket_id: str | None = None,
    year: int | None = None,
    limit: int = 20,
):
    """Full-text search over comment bodies, scoped to a single agency.

    Reads exactly one file — the canonical per-agency mirror — no partition
    discovery step needed. It's sorted by (docket_id, posted_date), so adding
    `docket_id` lets DuckDB prune most of the file via Parquet row-group
    stats; `year` (translated to a posted_date range, since the canonical
    tree has no year/month columns) narrows the text scan further.
    """
    filters = []
    if docket_id:
        filters.append(f"docket_id = '{docket_id}'")
    if year:
        filters.append(f"posted_date >= '{year}-01-01' AND posted_date < '{year + 1}-01-01'")
    filters.append(f"comment ILIKE '%{phrase}%'")

    url = f"{R2_BASE_URL}/comments/agency/agency_code={agency}/part-0.parquet"
    return conn.execute(f"""
        SELECT comment_id, docket_id, posted_date,
               substr(comment, 1, 240) AS preview
        FROM read_parquet('{url}')
        WHERE {' AND '.join(filters)}
        ORDER BY posted_date DESC
        LIMIT {limit}
    """).fetchdf()


# Example: narrow to one docket — cheapest, always safe.
search_comments("methane", agency="EPA", docket_id="EPA-HQ-OAR-2021-0317", limit=5)

,comment_id,docket_id,posted_date,preview
0,EPA-HQ-OAR-2021-0317-2890,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,"Dear Administrator Regan, <br/><br/>Thank you ..."
1,EPA-HQ-OAR-2021-0317-2888,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,I am deeply concerned about the impacts of air...
2,EPA-HQ-OAR-2021-0317-2891,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,I strongly support requiring oil and gas devel...
3,EPA-HQ-OAR-2021-0317-2887,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,"Dear EPA Administrator Regan,<br/><br/>Methane..."
4,EPA-HQ-OAR-2021-0317-2884,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,"Dear Environmental Protection Agency,<br/><br/..."


In [7]:
# Agency + year scope, no docket_id — still fast since only the comment
# and posted_date columns need reading for this one agency's file.
search_comments("forever chemicals", agency="EPA", year=2024, limit=5)

,comment_id,docket_id,posted_date,preview
0,EPA-HQ-OPPT-2024-0131-0055,EPA-HQ-OPPT-2024-0131,2024-12-31T05:00:00Z,"Sunday, December 29, 2024<br/><br/>Office of P..."
1,EPA-HQ-OPPT-2024-0131-0046,EPA-HQ-OPPT-2024-0131,2024-12-31T05:00:00Z,I strongly support the EPA&rsquo;s proposed re...
2,EPA-HQ-OPPT-2024-0131-0039,EPA-HQ-OPPT-2024-0131,2024-12-31T05:00:00Z,We must protect the public from &quot;forever ...
3,EPA-HQ-OPPT-2024-0131-0050,EPA-HQ-OPPT-2024-0131,2024-12-31T05:00:00Z,The science is clear&mdash;PFAS (per and poly-...
4,EPA-HQ-OPPT-2023-0538-0535,EPA-HQ-OPPT-2023-0538,2024-12-11T05:00:00Z,I strongly support the EPA&#39;s proposal to a...


### Tips

- **`docket_id` is the cheapest filter.** The canonical tree is sorted by `(docket_id, posted_date)`, so scoping to one docket lets DuckDB skip almost the whole file via Parquet row-group pruning — sub-2s even for a busy agency.
- **Big agencies are still fine unscoped, just slower.** A full `comment ILIKE` scan over the largest per-agency file (FWS, ~2.6M rows) takes a few seconds, not minutes — but add `docket_id` or `year` when you can.
- **`ILIKE` is substring, not token-aware.** For phrase matching with stemming/fuzziness, pull the matched rows into memory and run a real search lib (Whoosh, rank_bm25) locally.
- **Want ranking?** Add `regexp_matches(comment, ...)` or score by `len(regexp_extract_all(...))` — cheap and runs pushed-down inside DuckDB.
- **`comment` is the field to search, not `text_content`.** `text_content` (backfilled attachment text) is only populated on a small fraction of comments — see the data dictionary for current coverage — so treat it as a bonus, not the primary text.

## Bonus — the hosted UI

If you don't want to write code at all, the live site has a search box at **`/search`** backed by exactly the Layer 1 index above (via [MiniSearch](https://github.com/lucaong/minisearch) for fuzzy/prefix ranking). See `spicy-regs-ui/src/lib/search/index.ts` for the client implementation.

## Which layer to use?

- **I want the same search the site has** → Layer 1 (or just use the site).
- **I want SQL flexibility on docket metadata** → Layer 2.
- **I want to search inside comments (the mirrulations-search use case)** → Layer 3, always scoped by agency.

None of the above require running a server.